#AI Knowledge Graph Builder for Enterprise Intelligence

#ISRO Mission Navigator

##MODULE 5 — INTERACTIVE GRAPH DASHBOARD
1.   Visual Graph Exploration
2.   AI-powered Q&A over Graph
1.   Filtering & Drill-down
2.   Dual RAG System (Text + Graph)
1.   Colab Dashboard (Plotly)
2.   Full React App (D3.js + Neo4j GraphQL)
1.   Deploy to Vercel & Netlify


##CREATING INTERACTIVE DASHBOARD

###STEP 1 — Install Dependencies

In [8]:
!pip install neo4j plotly networkx pyvis langchain langchain-community

In [2]:
!pip install networkx plotly
!pip install pandas

###Load Graph (METHOD 1)

In [3]:
import networkx as nx
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("/content/exported_graph.csv")

G = nx.from_pandas_edgelist(
    df,
    source="source",
    target="target",
    edge_attr="relationship"
)

In [4]:
# Generate Interactive Visualization
pos = nx.spring_layout(G, k=0.3)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=1),
    mode='lines'
)

node_x = []
node_y = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=list(G.nodes()),
    marker=dict(size=8)
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(showlegend=False)
fig.show()

###STEP 2 — Connect to Neo4j

In [5]:
from neo4j import GraphDatabase

uri = "neo4j+s://5be8df30.databases.neo4j.io"        # or AuraDB URI
username = "neo4j"
password = "jdrjnjNfgcFfBUY7PpsTE7YK92kLYXYlrCYZVOPcBiA"

driver = GraphDatabase.driver(uri, auth=(username, password))

###STEP 3 — Fetch Graph Data

In [6]:
def fetch_graph():
    query = """
    MATCH (a)-[r]->(b)
    RETURN a.name AS source, type(r) AS relation, b.name AS target
    LIMIT 200
    """

    with driver.session() as session:
        result = session.run(query)
        return [record.data() for record in result]

graph_data = fetch_graph()

###STEP 4 — Build Network Graph (Premium Plotly)

In [7]:
import networkx as nx
import plotly.graph_objects as go

G = nx.DiGraph()

for row in graph_data:
    G.add_edge(row["source"], row["target"], label=row["relation"])

pos = nx.spring_layout(G, k=0.5)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

node_x = []
node_y = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1),
    hoverinfo='none',
    mode='lines'))

fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=list(G.nodes()),
    textposition="top center",
    marker=dict(size=20, color="blue")))

fig.update_layout(
    title="ISRO Knowledge Graph",
    showlegend=False,
    hovermode='closest')

fig.show()